In [ ]:
from pyspark.sql.functions import current_timestamp

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schema/volume/storage are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
catalog = dbutils.widgets.get("catalog")

# Raw/lossless bronze: Auto Loader stores the .pb bytes as-is. Identical shape to the trip-updates
# protobuf bronze — only the landing path and table differ. from_protobuf decode happens in silver.
# The vehicle-positions ingestion function must land files under this path (contract):
#   landing/vline_vehicle_positions/date=YYYYMMDD/vline_vp_*.pb
table_name = "vline_vehicle_positions"
table_path = f"{catalog}.`01_bronze`.{table_name}"
base       = "abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_vehicle_positions/"
checkpoint = f"/Volumes/{catalog}/01_bronze/_checkpoints/{table_name}"

bronze = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "binaryFile")
      .option("cloudFiles.schemaLocation", checkpoint + "/schema")
      .load(base)
      .withColumn("_ingest_ts", current_timestamp())
)

(bronze.writeStream
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable(table_path)
 )